# 1. 프로젝트 경로 및 데이터 불러오기

In [1]:

import sys
from pathlib import Path

import numpy as np
import pandas as pd

# 프로젝트 루트 설정
ROOT = Path.cwd().resolve().parents[1]

assert (ROOT / "src").is_dir(), f"프로젝트 경로 확인 필요: {ROOT}"

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import DATA_DIR
from src.data_loader import load_data, split_xy, encode_target
from src.validation import get_folds, twin_groups

# 데이터 불러오기
train, test = load_data(DATA_DIR)
X_raw, y_raw = split_xy(train)
y, le = encode_target(y_raw)

print("프로젝트 루트:", ROOT)
print("Train:", X_raw.shape)
print("클래스 수:", len(le.classes_))
print("데이터 로드 완료")


프로젝트 루트: /Users/admin/Documents/ah07-cancer-subtype
Train: (6201, 4384)
클래스 수: 26
데이터 로드 완료


In [2]:
# 팀 공통 Dev/Holdout 분할 불러오기


import importlib.util

# 팀 공통 고정 Holdout 분할 불러오기
manifest_path = ROOT / "scripts" / "fixed_holdout_manifest.py"

assert manifest_path.exists(), f"파일 없음: {manifest_path}"

spec = importlib.util.spec_from_file_location(
    "fixed_holdout_manifest",
    manifest_path
)
holdout_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(holdout_module)

# 기존 분할 검증
assert holdout_module.ROOT.resolve() == ROOT.resolve()

dev_idx, hold_idx = holdout_module.verify()

assert len(dev_idx) + len(hold_idx) == len(y)
assert len(np.intersect1d(dev_idx, hold_idx)) == 0

# Dev 데이터만 사용
X_dev_raw = X_raw.iloc[dev_idx].reset_index(drop=True)
y_dev = np.asarray(y)[dev_idx]

# 팀 공통 Fold 생성
team_folds = get_folds(
    y_dev,
    X=X_dev_raw,
    classes=le.classes_
)

print("Dev:", X_dev_raw.shape)
print("Holdout 크기:", len(hold_idx))
print("Fold 수:", len(team_folds))
print("클래스 수:", len(np.unique(y_dev)))


VERIFIED: dev=4960, holdout=1241; same IDs and row signatures
Dev: (4960, 4384)
Holdout 크기: 1241
Fold 수: 5
클래스 수: 26


# 2. 실험

##  실험 1. 환자별 변이 유형 비율·엔트로피 피처 생성

In [3]:
# 변이 데이터 형식 확인


# 변이가 기록된 값만 추출 (결측치 및 WT 제외)
values = X_dev_raw.stack().dropna().astype(str).str.strip()

mutation_values = values[
    ~values.str.upper().isin(["WT", "", "NAN", "NONE"])
]

print("전체 변이 기록 수:", len(mutation_values))
print("고유 변이 표기 수:", mutation_values.nunique())

print("\n[가장 많이 등장하는 변이 표기 TOP 20]")
display(mutation_values.value_counts().head(20))

print("\n[변이 표기 예시 30개]")
print(mutation_values.drop_duplicates().head(30).tolist())

print("\n[특수 표기 포함 여부]")
for pattern in ["fs", "*", "del", "ins", "splice", "X"]:
    count = mutation_values.str.contains(
        pattern, case=False, regex=False
    ).sum()
    print(f"{pattern}: {count:,}건")


전체 변이 기록 수: 175109
고유 변이 표기 수: 101766

[가장 많이 등장하는 변이 표기 TOP 20]


R132H     283
V600E     255
E545K     114
H1047R    105
R273C      66
E542K      54
R132C      52
R175H      50
R273H      48
R248W      40
R248Q      38
R213*      35
R196*      31
L42L       30
R282W      30
R80*       27
K16fs      26
Y220C      26
L45L       22
L159L      21
Name: count, dtype: int64


[변이 표기 예시 30개]
['D623D', 'F736C', 'P117R', 'R305S', 'N23K', 'L192V', 'G592G', 'K277K', 'A699T', 'L111L', 'I136M', 'A219T', 'E237*', 'A746D', 'D465D', 'N1445D', 'T218T', 'G764G', 'E660Q', 'E291A', 'I382M', 'V412A', 'P222R', 'R104H', 'P17R', 'T1115M', 'L1854fs', 'R36H', 'R69Q', 'A372T']

[특수 표기 포함 여부]
fs: 7,608건
*: 10,334건
del: 3건
ins: 0건
splice: 0건
X: 0건


In [4]:
# 변이 유형 분류 및 예외 확인

import re

def classify_mutation(value):
    s = str(value).strip()

    # 프레임시프트
    if "fs" in s.lower():
        return "frameshift"

    # 넌센스
    if "*" in s:
        return "nonsense"

    # 단일 아미노산 치환 또는 동의 변이
    match = re.fullmatch(r"([A-Z])(\d+)([A-Z])", s)

    if match:
        ref_aa, position, alt_aa = match.groups()

        if ref_aa == alt_aa:
            return "synonymous"
        return "missense"

    return "other"


mutation_types = mutation_values.map(classify_mutation)

print("[변이 유형별 기록 수]")
display(mutation_types.value_counts())

print("\n[변이 유형별 비율]")
display(
    (mutation_types.value_counts(normalize=True) * 100)
    .round(2)
)

print("\n[기타 변이 표기 TOP 30]")
other_values = mutation_values[mutation_types == "other"]
display(other_values.value_counts().head(30))

print("\n기타 변이 고유 표기 수:", other_values.nunique())


[변이 유형별 기록 수]


missense      101987
synonymous     40555
other          14701
nonsense       10258
frameshift      7608
Name: count, dtype: int64


[변이 유형별 비율]


missense      58.24
synonymous    23.16
other          8.40
nonsense       5.86
frameshift     4.34
Name: proportion, dtype: float64


[기타 변이 표기 TOP 30]


L26V L24V                      15
V600E V600E                    10
S8S S6A                         8
S43G P45P                       6
E726K H1047R                    4
S372S K373E                     4
C19W R5R                        4
P534P E556K S557T               3
E545K E726K                     3
E453K E545K                     3
S183F S183F                     3
L279L L279L                     3
P11S P11S                       3
P80L P80L                       3
E28K E28K                       3
G73G G81G                       3
E556K S557T                     3
V241L R245L                     2
312_313QY>HH                    2
R175H T155P                     2
L501V C500R                     2
V1325L G184V                    2
S185I Q186K                     2
P1109L R1107Q                   2
D28G L89H                       2
K1104E G1102E V1105L P1107P     2
G136S I174V                     2
A1061V I1056I                   2
N381N I378I                     2
82_83DA>EI    


기타 변이 고유 표기 수: 14504


In [5]:
# 다중 변이 표기 구조 확인


# 하나의 유전자 셀에 여러 변이가 있는 경우 분리
split_mutations = mutation_values.str.split().explode()

# 분리된 변이를 기존 함수로 다시 분류
split_types = split_mutations.map(classify_mutation)

print("기존 변이 기록 수:", len(mutation_values))
print("분리 후 변이 기록 수:", len(split_mutations))

print("\n[분리 후 변이 유형별 기록 수]")
display(split_types.value_counts())

print("\n[분리 후 변이 유형별 비율 (%)]")
display(
    (split_types.value_counts(normalize=True) * 100).round(2)
)

print("\n[여전히 분류되지 않는 변이 TOP 20]")
remaining_other = split_mutations[split_types == "other"]
display(remaining_other.value_counts().head(20))

print("\n분류되지 않는 변이 수:", len(remaining_other))
print("분류되지 않는 고유 표기 수:", remaining_other.nunique())


기존 변이 기록 수: 175109
분리 후 변이 기록 수: 205415

[분리 후 변이 유형별 기록 수]


missense      132806
synonymous     53810
nonsense       10764
frameshift      7899
other            136
Name: count, dtype: int64


[분리 후 변이 유형별 비율 (%)]


missense      64.65
synonymous    26.20
nonsense       5.24
frameshift     3.85
other          0.07
Name: proportion, dtype: float64


[여전히 분류되지 않는 변이 TOP 20]


312_313QY>HH      2
82_83DA>EI        2
929_930LL>LL      2
435_436EA>DS      2
711_712FL>FL      2
594_595ME>IK      2
1105_1106PL>PL    2
187_188PL>PL      2
1441_1442AL>AL    2
203_204LL>LL      2
483_484MP>IA      2
1529_1530AL>AL    1
142_143RL>RL      1
4510_4511AL>AL    1
91_92NH>KY        1
123_124MP>IA      1
1762_1763QG>HC    1
164_165KA>NS      1
569_570HG>QW      1
1181_1182ML>IV    1
Name: count, dtype: int64


분류되지 않는 변이 수: 136
분류되지 않는 고유 표기 수: 125


In [6]:
# 환자별 변이 유형 피처 생성


# 환자별 변이 유형 집계
mutation_long = (
    X_dev_raw
    .stack()
    .rename("mutation")
    .reset_index()
    .rename(columns={"level_0": "patient_idx", "level_1": "gene"})
)

# WT 및 빈 값 제외
mutation_long["mutation"] = (
    mutation_long["mutation"].astype(str).str.strip()
)

mutation_long = mutation_long[
    ~mutation_long["mutation"].str.upper().isin(
        ["WT", "", "NAN", "NONE"]
    )
].copy()

# 공백으로 구분된 다중 변이 분리
mutation_long["mutation"] = mutation_long["mutation"].str.split()
mutation_long = mutation_long.explode("mutation")

# 개별 변이 유형 분류
mutation_long["type"] = (
    mutation_long["mutation"].map(classify_mutation)
)

type_names = [
    "missense", "synonymous", "nonsense",
    "frameshift", "other"
]

# 환자별 유형 개수
type_counts = pd.crosstab(
    mutation_long["patient_idx"],
    mutation_long["type"]
).reindex(
    index=range(len(X_dev_raw)),
    columns=type_names,
    fill_value=0
)

# 환자별 전체 변이 수
total_counts = type_counts.sum(axis=1)

# 유형별 비율
type_ratios = type_counts.div(
    total_counts.replace(0, np.nan),
    axis=0
).fillna(0)

type_ratios.columns = [
    f"mutation_ratio_{name}"
    for name in type_ratios.columns
]

# 섀넌 엔트로피: -sum(p * log2(p))
p = type_ratios.to_numpy(dtype=float)

entropy = -np.sum(
    np.where(p > 0, p * np.log2(np.clip(p, 1e-12, None)), 0),
    axis=1
)

# 최종 피처
X_type_features = type_ratios.copy()

X_type_features["mutation_type_entropy"] = entropy
X_type_features["mutation_type_diversity"] = (
    (type_counts > 0).sum(axis=1)
)
X_type_features["synonymous_only"] = (
    (type_counts["synonymous"] > 0)
    & (type_counts["synonymous"] == total_counts)
).astype(int)

X_type_features = X_type_features.astype(np.float32)

print("추가 피처 크기:", X_type_features.shape)
print("\n[추가 피처 목록]")
print(X_type_features.columns.tolist())

print("\n[기초 통계]")
display(X_type_features.describe().round(3))

print("\n결측치 수:", X_type_features.isna().sum().sum())


추가 피처 크기: (4960, 8)

[추가 피처 목록]
['mutation_ratio_missense', 'mutation_ratio_synonymous', 'mutation_ratio_nonsense', 'mutation_ratio_frameshift', 'mutation_ratio_other', 'mutation_type_entropy', 'mutation_type_diversity', 'synonymous_only']

[기초 통계]


,mutation_ratio_missense,mutation_ratio_synonymous,mutation_ratio_nonsense,mutation_ratio_frameshift,mutation_ratio_other,mutation_type_entropy,mutation_type_diversity,synonymous_only
count,4960.000,4960.000,4960.000,4960.000,4960.000,4960.000,4960.000,4960.000
mean,0.657,0.226,0.052,0.065,0.001,1.076,2.936,0.004
std,0.175,0.146,0.085,0.113,0.009,0.434,0.963,0.065
min,0.000,0.000,0.000,0.000,0.000,-0.000,1.000,0.000
25%,0.571,0.143,0.000,0.000,0.000,0.918,2.000,0.000
50%,0.667,0.228,0.027,0.017,0.000,1.157,3.000,0.000
75%,0.750,0.308,0.077,0.095,0.000,1.371,4.000,0.000
max,1.000,1.000,1.000,1.000,0.500,2.197,5.000,1.000



결측치 수: 0


In [9]:
# 기존 피처 구성방법 E(E: 아미노산 치환 쌍 + 유전자별 끊김형) + 신규 8개 피처


from scipy.sparse import csr_matrix, hstack

from src.preprocessing import binarize_mutations
from src.features import (
    mutation_count,
    mutation_type_counts,
    amino_acid_pair_counts,
    gene_truncation_flags
)

# 기존 피처 구성방법 E
def make_E(X):
    parts = [
        binarize_mutations(X),
        np.log1p(mutation_count(X)),
        np.log1p(mutation_type_counts(X)),
        np.log1p(amino_acid_pair_counts(X)),
        gene_truncation_flags(X),
    ]

    out = pd.concat(parts, axis=1).astype(np.float32)

    assert not out.columns.has_duplicates
    assert out.index.equals(X.index)
    assert np.isfinite(out.to_numpy()).all()

    return out


# 기존 E 피처
X_dev_E = make_E(X_dev_raw)
X_E_sparse = csr_matrix(X_dev_E.to_numpy(dtype=np.float32))

# 신규 피처 추가
X_type_sparse = csr_matrix(
    X_type_features.to_numpy(dtype=np.float32)
)

X_E_plus_type = hstack(
    [X_E_sparse, X_type_sparse],
    format="csr"
)

assert X_E_sparse.shape == (4960, 9174)
assert X_E_plus_type.shape == (4960, 9182)

print("기존 E 피처:", X_E_sparse.shape)
print("E + 변이 유형 피처:", X_E_plus_type.shape)
print("추가된 피처 수:", X_E_plus_type.shape[1] - X_E_sparse.shape[1])


기존 E 피처: (4960, 9174)
E + 변이 유형 피처: (4960, 9182)
추가된 피처 수: 8


In [10]:
# 기존 E vs E+8 피처 OOF 비교


from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
import time

base_model = LogisticRegression(
    C=0.1,
    class_weight="balanced",
    solver="lbfgs",
    max_iter=3000
)

feature_sets = {
    "E_baseline": X_E_sparse,
    "E_plus_type8": X_E_plus_type
}

experiment_results = []
experiment_oof = {}
experiment_fold_scores = {}

for experiment_name, X_features in feature_sets.items():
    start = time.time()
    oof_pred = np.full(len(y_dev), -1, dtype=int)
    fold_scores = []

    for fold_num, (train_idx, val_idx) in enumerate(team_folds, start=1):
        model = clone(base_model)

        model.fit(
            X_features[train_idx],
            y_dev[train_idx]
        )

        pred = model.predict(X_features[val_idx])
        oof_pred[val_idx] = pred

        fold_f1 = f1_score(
            y_dev[val_idx],
            pred,
            average="macro",
            zero_division=0
        )
        fold_scores.append(fold_f1)

        print(
            f"{experiment_name} | "
            f"Fold {fold_num}/5 | "
            f"Macro F1: {fold_f1:.4f}"
        )

    assert np.all(oof_pred >= 0)

    oof_f1 = f1_score(
        y_dev,
        oof_pred,
        average="macro",
        zero_division=0
    )

    experiment_oof[experiment_name] = oof_pred
    experiment_fold_scores[experiment_name] = fold_scores

    experiment_results.append({
        "실험": experiment_name,
        "OOF Macro F1": oof_f1,
        "Fold 평균": np.mean(fold_scores),
        "Fold 표준편차": np.std(fold_scores),
        "실행 시간(초)": round(time.time() - start, 1)
    })

results_type8 = pd.DataFrame(experiment_results)

display(results_type8.round(5))

baseline_f1 = results_type8.loc[
    results_type8["실험"] == "E_baseline",
    "OOF Macro F1"
].iloc[0]

new_f1 = results_type8.loc[
    results_type8["실험"] == "E_plus_type8",
    "OOF Macro F1"
].iloc[0]

print(f"\nOOF 변화량: {new_f1 - baseline_f1:+.5f}")


E_baseline | Fold 1/5 | Macro F1: 0.5049
E_baseline | Fold 2/5 | Macro F1: 0.4632
E_baseline | Fold 3/5 | Macro F1: 0.4740
E_baseline | Fold 4/5 | Macro F1: 0.4757
E_baseline | Fold 5/5 | Macro F1: 0.4903
E_plus_type8 | Fold 1/5 | Macro F1: 0.5077
E_plus_type8 | Fold 2/5 | Macro F1: 0.4561
E_plus_type8 | Fold 3/5 | Macro F1: 0.4726
E_plus_type8 | Fold 4/5 | Macro F1: 0.4707
E_plus_type8 | Fold 5/5 | Macro F1: 0.4917


,실험,OOF Macro F1,Fold 평균,Fold 표준편차,실행 시간(초)
0,E_baseline,0.48485,0.48160,0.01448,13.7
1,E_plus_type8,0.48323,0.47975,0.01797,18.5



OOF 변화량: -0.00162


In [11]:

# 기존 E와 E+8의 암종별 F1 비교
class_ids = np.arange(len(le.classes_))

baseline_class_f1 = f1_score(
    y_dev,
    experiment_oof["E_baseline"],
    labels=class_ids,
    average=None,
    zero_division=0
)

new_class_f1 = f1_score(
    y_dev,
    experiment_oof["E_plus_type8"],
    labels=class_ids,
    average=None,
    zero_division=0
)

class_comparison = pd.DataFrame({
    "암종": le.classes_,
    "기존 E F1": baseline_class_f1,
    "E+8 F1": new_class_f1,
    "변화량": new_class_f1 - baseline_class_f1
})

print("[F1 개선 상위 10개 암종]")
display(
    class_comparison
    .sort_values("변화량", ascending=False)
    .head(10)
    .round(4)
)

print("\n[F1 하락 상위 10개 암종]")
display(
    class_comparison
    .sort_values("변화량")
    .head(10)
    .round(4)
)


[F1 개선 상위 10개 암종]


,암종,기존 E F1,E+8 F1,변화량
24,THYM,0.1581,0.1769,0.0188
1,BLCA,0.4485,0.4596,0.0111
12,LIHC,0.4427,0.4494,0.0067
14,LUSC,0.5603,0.5654,0.0051
7,HNSC,0.3673,0.3699,0.0026
3,CESC,0.4073,0.4086,0.0013
2,BRCA,0.5357,0.5364,0.0007
22,TGCT,0.5282,0.5286,0.0004
10,LAML,0.5481,0.5483,0.0001
5,DLBC,0.3182,0.3182,0.0000



[F1 하락 상위 10개 암종]


,암종,기존 E F1,E+8 F1,변화량
0,ACC,0.8571,0.8348,-0.0224
13,LUAD,0.4719,0.4519,-0.0201
11,LGG,0.5660,0.5567,-0.0092
18,PRAD,0.3364,0.3274,-0.0090
9,KIRC,0.4024,0.3959,-0.0066
15,OV,0.4348,0.4299,-0.0049
19,SARC,0.1733,0.1702,-0.0031
17,PCPG,0.3077,0.3053,-0.0023
6,GBMLGG,0.4340,0.4318,-0.0022
25,UCEC,0.6316,0.6296,-0.0019


## 실험 2. 유전자 내 다중 히트 피처 생성

In [12]:

# 기존 mutation_long에는 개별 변이로 분리된 기록이 들어 있음

# 환자 × 유전자별 변이 개수
gene_hit_counts = (
    mutation_long
    .groupby(["patient_idx", "gene"])
    .size()
    .rename("hit_count")
    .reset_index()
)

# 환자별 다중 히트 유전자 수
multi_hit_gene_count = (
    gene_hit_counts
    .assign(is_multi_hit=lambda df: df["hit_count"] >= 2)
    .groupby("patient_idx")["is_multi_hit"]
    .sum()
    .reindex(range(len(X_dev_raw)), fill_value=0)
)

# 환자별 한 유전자에서의 최대 히트 수
max_hits_per_gene = (
    gene_hit_counts
    .groupby("patient_idx")["hit_count"]
    .max()
    .reindex(range(len(X_dev_raw)), fill_value=0)
)

X_multihit_features = pd.DataFrame({
    "multi_hit_gene_count": multi_hit_gene_count,
    "max_hits_per_gene": max_hits_per_gene
}).astype(np.float32)

assert X_multihit_features.shape == (4960, 2)
assert X_multihit_features.isna().sum().sum() == 0

print("추가 피처 크기:", X_multihit_features.shape)

print("\n[기초 통계]")
display(X_multihit_features.describe().round(3))

print("\n[다중 히트 유전자가 있는 환자]")
print(
    (X_multihit_features["multi_hit_gene_count"] > 0).sum(),
    "명"
)

print("\n[최대 히트 수 분포]")
display(
    X_multihit_features["max_hits_per_gene"]
    .value_counts()
    .sort_index()
    .head(15)
)


추가 피처 크기: (4960, 2)

[기초 통계]


,multi_hit_gene_count,max_hits_per_gene
count,4960.000,4960.000
mean,3.591,1.551
std,40.245,1.515
min,0.000,1.000
25%,0.000,1.000
50%,0.000,1.000
75%,1.000,2.000
max,2021.000,64.000



[다중 히트 유전자가 있는 환자]
1679 명

[최대 히트 수 분포]


max_hits_per_gene
1.0     3281
2.0     1231
3.0      264
4.0       89
5.0       29
6.0       23
7.0       11
8.0        7
9.0        3
10.0       1
11.0       5
12.0       5
13.0       2
14.0       1
15.0       2
Name: count, dtype: int64

In [13]:
# 다중 히트 피처 이상치 확인


# 다중 히트 유전자 수가 많은 환자 TOP 10
top_patients = (
    X_multihit_features
    .sort_values("multi_hit_gene_count", ascending=False)
    .head(10)
    .copy()
)

# 전체 변이 수 및 변이 기록 유전자 수 추가
top_patients["total_mutations"] = total_counts.loc[
    top_patients.index
].values

top_patients["mutated_genes"] = (
    gene_hit_counts
    .groupby("patient_idx")
    .size()
    .reindex(top_patients.index, fill_value=0)
    .values
)

top_patients["cancer_type"] = le.inverse_transform(
    y_dev[top_patients.index.to_numpy()]
)

print("[다중 히트 유전자 수 TOP 10 환자]")
display(top_patients)

# 최댓값 환자의 실제 유전자별 히트 수 확인
top_patient_idx = int(top_patients.index[0])

print(f"\n[환자 {top_patient_idx}의 유전자별 히트 수 TOP 10]")
display(
    gene_hit_counts[
        gene_hit_counts["patient_idx"] == top_patient_idx
    ]
    .sort_values("hit_count", ascending=False)
    .head(10)
)

print("\n[다중 히트 유전자 수 분위수]")
display(
    X_multihit_features["multi_hit_gene_count"]
    .quantile([0.5, 0.9, 0.95, 0.99, 0.999, 1.0])
)


[다중 히트 유전자 수 TOP 10 환자]


,multi_hit_gene_count,max_hits_per_gene,total_mutations,mutated_genes,cancer_type
patient_idx,,,,,
1126,2021.0,64.0,9972,2025,SKCM
1453,1144.0,24.0,4768,2393,PAAD
1430,646.0,12.0,2773,1754,UCEC
3278,590.0,17.0,2752,1631,COAD
3982,541.0,12.0,2459,1600,UCEC
4033,457.0,16.0,2222,1482,UCEC
984,432.0,15.0,2131,1442,UCEC
3293,425.0,11.0,1979,1288,UCEC
4909,359.0,14.0,1900,1294,UCEC



[환자 1126의 유전자별 히트 수 TOP 10]


,patient_idx,gene,hit_count
43403,1126,PCLO,64
43656,1126,RYR1,60
43669,1126,SCN10A,44
42482,1126,COL5A3,39
43820,1126,SPTA1,38
43019,1126,KALRN,36
43780,1126,SLIT3,36
43092,1126,LCT,32
43315,1126,NOS1,32
42298,1126,C3,30



[다중 히트 유전자 수 분위수]


0.500       0.000
0.900       3.000
0.950       8.000
0.990      44.820
0.999     460.444
1.000    2021.000
Name: multi_hit_gene_count, dtype: float64

In [14]:
# 다중 히트 피처 추가 및 성능 비교


# 다중 히트 피처 log1p 변환
X_multihit_log = np.log1p(X_multihit_features).astype(np.float32)

# 기존 E + 다중 히트 피처 2개
X_multihit_sparse = csr_matrix(
    X_multihit_log.to_numpy(dtype=np.float32)
)

X_E_plus_multihit = hstack(
    [X_E_sparse, X_multihit_sparse],
    format="csr"
)

assert X_E_plus_multihit.shape == (4960, 9176)

print("기존 E:", X_E_sparse.shape)
print("E + 다중 히트 2개:", X_E_plus_multihit.shape)

# 기존과 동일한 LR 및 팀 공통 Fold 사용
oof_pred = np.full(len(y_dev), -1, dtype=int)
multihit_fold_scores = []

for fold_num, (train_idx, val_idx) in enumerate(team_folds, start=1):
    model = clone(base_model)

    model.fit(
        X_E_plus_multihit[train_idx],
        y_dev[train_idx]
    )

    pred = model.predict(X_E_plus_multihit[val_idx])
    oof_pred[val_idx] = pred

    fold_f1 = f1_score(
        y_dev[val_idx],
        pred,
        average="macro",
        zero_division=0
    )
    multihit_fold_scores.append(fold_f1)

    print(f"Fold {fold_num}/5: {fold_f1:.5f}")

assert np.all(oof_pred >= 0)

multihit_oof_f1 = f1_score(
    y_dev,
    oof_pred,
    average="macro",
    zero_division=0
)

experiment_oof["E_plus_multihit2"] = oof_pred
experiment_fold_scores["E_plus_multihit2"] = multihit_fold_scores

print("\n[OOF Macro F1 비교]")
print(f"기존 E:          {baseline_f1:.5f}")
print(f"E + 다중 히트:   {multihit_oof_f1:.5f}")
print(f"변화량:          {multihit_oof_f1 - baseline_f1:+.5f}")


기존 E: (4960, 9174)
E + 다중 히트 2개: (4960, 9176)
Fold 1/5: 0.51309
Fold 2/5: 0.46148
Fold 3/5: 0.47160
Fold 4/5: 0.48974
Fold 5/5: 0.48700

[OOF Macro F1 비교]
기존 E:          0.48485
E + 다중 히트:   0.48774
변화량:          +0.00289


## 실험 3. 유전자 × 미스센스 피처 생성

In [15]:

# 미스센스 변이가 기록된 환자-유전자 조합
missense_records = (
    mutation_long.loc[
        mutation_long["type"] == "missense",
        ["patient_idx", "gene"]
    ]
    .drop_duplicates()
)

print("미스센스 환자-유전자 조합 수:", len(missense_records))
print("미스센스가 관찰된 유전자 수:", missense_records["gene"].nunique())

print("\n[미스센스 변이 환자가 많은 유전자 TOP 20]")
display(
    missense_records["gene"]
    .value_counts()
    .head(20)
)

print("\n[환자별 미스센스 변이 유전자 수]")
display(
    missense_records
    .groupby("patient_idx")
    .size()
    .reindex(range(len(X_dev_raw)), fill_value=0)
    .describe()
    .round(2)
)


미스센스 환자-유전자 조합 수: 117342
미스센스가 관찰된 유전자 수: 4217

[미스센스 변이 환자가 많은 유전자 TOP 20]


gene
TP53      1049
PIK3CA     526
RYR2       404
PCLO       372
SYNE1      368
IDH1       358
BRAF       355
SPTA1      280
RYR1       255
DMD        250
DST        237
RELN       204
PKHD1      198
AHNAK      192
MXRA5      192
COL6A3     190
KMT2D      185
HUWE1      182
FBN2       182
PTPRD      181
Name: count, dtype: int64


[환자별 미스센스 변이 유전자 수]


count    4960.00
mean       23.66
std        74.13
min         0.00
25%         4.00
50%         9.00
75%        18.00
max      1939.00
dtype: float64

In [16]:
# Fold별 상위 500개 유전자 피처 생성 함수


from scipy.sparse import csr_matrix

# 환자 × 유전자 미스센스 여부 행렬
missense_binary = pd.crosstab(
    missense_records["patient_idx"],
    missense_records["gene"]
)

missense_binary = (
    missense_binary
    .gt(0)
    .astype(np.float32)
    .reindex(index=range(len(X_dev_raw)), fill_value=0)
)

def make_top500_missense(train_idx, val_idx):
    # 학습 Fold에서만 유전자별 미스센스 환자 수 계산
    train_counts = missense_binary.iloc[train_idx].sum(axis=0)

    # 빈도 내림차순, 동률은 유전자명 오름차순
    top_genes = (
        train_counts
        .rename("count")
        .rename_axis("gene")
        .reset_index()
        .sort_values(
            ["count", "gene"],
            ascending=[False, True]
        )
        .head(500)["gene"]
        .tolist()
    )

    # 학습/검증에 동일한 500개 유전자 적용
    X_train_extra = csr_matrix(
        missense_binary.iloc[train_idx][top_genes]
        .to_numpy(dtype=np.float32)
    )

    X_val_extra = csr_matrix(
        missense_binary.iloc[val_idx][top_genes]
        .to_numpy(dtype=np.float32)
    )

    return X_train_extra, X_val_extra, top_genes


# 첫 번째 Fold에서 동작 확인
train_idx, val_idx = team_folds[0]

X_train_extra, X_val_extra, top_genes = make_top500_missense(
    train_idx, val_idx
)

print("학습 Fold 추가 피처:", X_train_extra.shape)
print("검증 Fold 추가 피처:", X_val_extra.shape)
print("선정된 유전자 수:", len(top_genes))
print("상위 10개 유전자:", top_genes[:10])


학습 Fold 추가 피처: (3968, 500)
검증 Fold 추가 피처: (992, 500)
선정된 유전자 수: 500
상위 10개 유전자: ['TP53', 'PIK3CA', 'RYR2', 'PCLO', 'SYNE1', 'BRAF', 'IDH1', 'SPTA1', 'DMD', 'RYR1']


In [ ]:
# E + 미스센스 피처 500개 성능 비교


# 실험 3: E + 유전자별 미스센스 여부 500개

oof_missense = np.full(len(y_dev), -1, dtype=int)
missense_fold_scores = []
selected_genes_by_fold = {}

for fold_num, (train_idx, val_idx) in enumerate(team_folds, start=1):

    # 학습 Fold에서만 상위 500개 유전자 선정
    X_train_extra, X_val_extra, top_genes = make_top500_missense(
        train_idx, val_idx
    )

    selected_genes_by_fold[fold_num] = top_genes

    # 기존 E + 미스센스 피처
    X_train = hstack(
        [X_E_sparse[train_idx], X_train_extra],
        format="csr"
    )
    X_val = hstack(
        [X_E_sparse[val_idx], X_val_extra],
        format="csr"
    )

    assert X_train.shape[1] == 9674
    assert X_val.shape[1] == 9674

    model = clone(base_model)
    model.fit(X_train, y_dev[train_idx])

    pred = model.predict(X_val)
    oof_missense[val_idx] = pred

    fold_f1 = f1_score(
        y_dev[val_idx],
        pred,
        average="macro",
        zero_division=0
    )
    missense_fold_scores.append(fold_f1)

    print(f"Fold {fold_num}/5: {fold_f1:.5f}")

assert np.all(oof_missense >= 0)

missense_oof_f1 = f1_score(
    y_dev,
    oof_missense,
    average="macro",
    zero_division=0
)

experiment_oof["E_plus_missense500"] = oof_missense
experiment_fold_scores["E_plus_missense500"] = missense_fold_scores

print("\n[OOF Macro F1 비교]")
print(f"기존 E:             {baseline_f1:.5f}")
print(f"E + 미스센스 500:   {missense_oof_f1:.5f}")
print(f"변화량:             {missense_oof_f1 - baseline_f1:+.5f}")

print("\n[실험 1~3 비교]")
display(pd.DataFrame({
    "실험": [
        "기존 E",
        "변이 유형 8개",
        "다중 히트 2개",
        "미스센스 500개"
    ],
    "OOF Macro F1": [
        baseline_f1,
        new_f1,
        multihit_oof_f1,
        missense_oof_f1
    ]
}).round(5))


Fold 1/5: 0.49880
Fold 2/5: 0.45764
Fold 3/5: 0.45248
Fold 4/5: 0.48739
Fold 5/5: 0.49339

[OOF Macro F1 비교]
기존 E:             0.48485
E + 미스센스 500:   0.48079
변화량:             -0.00406

[실험 1~3 비교]


,실험,OOF Macro F1
0,기존 E,0.48485
1,변이 유형 8개,0.48323
2,다중 히트 2개,0.48774
3,미스센스 500개,0.48079


# 3. XGB 검증

## 3-1. 검증 준비

In [22]:

from xgboost import XGBClassifier
from src.config import SEED

K = len(le.classes_)

XGB_PARAMS = dict(
    n_estimators=250,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    num_class=K,
    eval_metric="mlogloss",
    tree_method="hist",
    device="cpu",
    random_state=SEED,
    n_jobs=-1
)

xgb_model = XGBClassifier(**XGB_PARAMS)

def xgb_sample_weights(y_train):
    labels = np.asarray(y_train, dtype=int)
    counts = np.bincount(labels, minlength=K)

    assert np.all(counts > 0)

    weights = 1.0 / np.sqrt(counts[labels])
    return (weights / weights.mean()).astype(np.float32)

print("XGB 모델 설정 완료")
print("클래스 수:", K)
print("Device:", XGB_PARAMS["device"])
print("Estimators:", XGB_PARAMS["n_estimators"])
print("Max depth:", XGB_PARAMS["max_depth"])
print("Seed:", SEED)


XGB 모델 설정 완료
클래스 수: 26
Device: cpu
Estimators: 250
Max depth: 3
Seed: 42


## 실험 1: 기존 E vs E + 변이 유형 비율·엔트로피 등 8개

In [23]:

import time

xgb_feature_sets = {
    "E_baseline": X_E_sparse,
    "E_plus_type8": X_E_plus_type
}

xgb_oof = {}
xgb_fold_scores = {}
xgb_results = []

for name, X_features in xgb_feature_sets.items():
    start = time.time()
    oof_pred = np.full(len(y_dev), -1, dtype=int)
    fold_scores = []

    print(f"\n[{name}]")

    for fold_num, (train_idx, val_idx) in enumerate(team_folds, start=1):
        model = clone(xgb_model)

        weights = xgb_sample_weights(y_dev[train_idx])

        model.fit(
            X_features[train_idx],
            y_dev[train_idx],
            sample_weight=weights
        )

        pred = model.predict(X_features[val_idx])
        oof_pred[val_idx] = pred

        score = f1_score(
            y_dev[val_idx],
            pred,
            average="macro",
            zero_division=0
        )
        fold_scores.append(score)

        print(f"Fold {fold_num}/5: {score:.5f}")

    assert np.all(oof_pred >= 0)

    oof_f1 = f1_score(
        y_dev,
        oof_pred,
        average="macro",
        zero_division=0
    )

    xgb_oof[name] = oof_pred
    xgb_fold_scores[name] = fold_scores

    xgb_results.append({
        "실험": name,
        "OOF Macro F1": oof_f1,
        "Fold 평균": np.mean(fold_scores),
        "Fold 표준편차": np.std(fold_scores),
        "실행 시간(초)": round(time.time() - start, 1)
    })

results_xgb = pd.DataFrame(xgb_results)

print("\n[XGB 실험 1 결과]")
display(results_xgb.round(5))

baseline_score = results_xgb.loc[
    results_xgb["실험"] == "E_baseline",
    "OOF Macro F1"
].iloc[0]

type8_score = results_xgb.loc[
    results_xgb["실험"] == "E_plus_type8",
    "OOF Macro F1"
].iloc[0]

print(f"\nOOF 변화량: {type8_score - baseline_score:+.5f}")



[E_baseline]
Fold 1/5: 0.49945
Fold 2/5: 0.45525
Fold 3/5: 0.47387
Fold 4/5: 0.47361
Fold 5/5: 0.47492

[E_plus_type8]
Fold 1/5: 0.48661
Fold 2/5: 0.45029
Fold 3/5: 0.44708
Fold 4/5: 0.46668
Fold 5/5: 0.47641

[XGB 실험 1 결과]


,실험,OOF Macro F1,Fold 평균,Fold 표준편차,실행 시간(초)
0,E_baseline,0.47917,0.47542,0.01407,65.8
1,E_plus_type8,0.46900,0.46541,0.01508,65.2



OOF 변화량: -0.01017


## 실험 2: 기존 E vs E + 다중 히트 2개

In [24]:

# XGB 실험 2: E + 다중 히트 2개

start = time.time()

name = "E_plus_multihit2"
X_features = X_E_plus_multihit

oof_pred = np.full(len(y_dev), -1, dtype=int)
fold_scores = []

for fold_num, (train_idx, val_idx) in enumerate(team_folds, start=1):
    model = clone(xgb_model)

    weights = xgb_sample_weights(y_dev[train_idx])

    model.fit(
        X_features[train_idx],
        y_dev[train_idx],
        sample_weight=weights
    )

    pred = model.predict(X_features[val_idx])
    oof_pred[val_idx] = pred

    score = f1_score(
        y_dev[val_idx],
        pred,
        average="macro",
        zero_division=0
    )
    fold_scores.append(score)

    print(f"Fold {fold_num}/5: {score:.5f}")

assert np.all(oof_pred >= 0)

multihit_xgb_f1 = f1_score(
    y_dev,
    oof_pred,
    average="macro",
    zero_division=0
)

xgb_oof[name] = oof_pred
xgb_fold_scores[name] = fold_scores

baseline_xgb_f1 = f1_score(
    y_dev,
    xgb_oof["E_baseline"],
    average="macro",
    zero_division=0
)

delta = multihit_xgb_f1 - baseline_xgb_f1

print("\n[XGB 실험 2 결과]")
print(f"기존 E:            {baseline_xgb_f1:.5f}")
print(f"E + 다중 히트 2개: {multihit_xgb_f1:.5f}")
print(f"변화량:            {delta:+.5f}")
print(f"실행 시간:         {time.time() - start:.1f}초")
print(f"+0.003 기준 충족:  {delta >= 0.003}")


Fold 1/5: 0.49341
Fold 2/5: 0.46001
Fold 3/5: 0.46488
Fold 4/5: 0.48666
Fold 5/5: 0.48941

[XGB 실험 2 결과]
기존 E:            0.47917
E + 다중 히트 2개: 0.48317
변화량:            +0.00400
실행 시간:         64.0초
+0.003 기준 충족:  True


## 실험 3: 기존 E vs E + 유전자별 미스센스 500개

In [25]:

# XGB 실험 3: E + 유전자별 미스센스 500개

start = time.time()

name = "E_plus_missense500"
oof_pred = np.full(len(y_dev), -1, dtype=int)
fold_scores = []

for fold_num, (train_idx, val_idx) in enumerate(team_folds, start=1):

    # 해당 Fold의 학습 데이터에서만 상위 500개 선정
    X_train_extra, X_val_extra, top_genes = make_top500_missense(
        train_idx, val_idx
    )

    # 기존 E + 미스센스 500개
    X_train = hstack(
        [X_E_sparse[train_idx], X_train_extra],
        format="csr"
    )
    X_val = hstack(
        [X_E_sparse[val_idx], X_val_extra],
        format="csr"
    )

    assert X_train.shape[1] == 9674
    assert X_val.shape[1] == 9674

    model = clone(xgb_model)

    weights = xgb_sample_weights(y_dev[train_idx])

    model.fit(
        X_train,
        y_dev[train_idx],
        sample_weight=weights
    )

    pred = model.predict(X_val)
    oof_pred[val_idx] = pred

    score = f1_score(
        y_dev[val_idx],
        pred,
        average="macro",
        zero_division=0
    )
    fold_scores.append(score)

    print(f"Fold {fold_num}/5: {score:.5f}")

assert np.all(oof_pred >= 0)

missense_xgb_f1 = f1_score(
    y_dev,
    oof_pred,
    average="macro",
    zero_division=0
)

xgb_oof[name] = oof_pred
xgb_fold_scores[name] = fold_scores

delta = missense_xgb_f1 - baseline_xgb_f1

print("\n[XGB 실험 3 결과]")
print(f"기존 E:             {baseline_xgb_f1:.5f}")
print(f"E + 미스센스 500개: {missense_xgb_f1:.5f}")
print(f"변화량:             {delta:+.5f}")
print(f"실행 시간:          {time.time() - start:.1f}초")
print(f"+0.003 기준 충족:   {delta >= 0.003}")


Fold 1/5: 0.49964
Fold 2/5: 0.45566
Fold 3/5: 0.46520
Fold 4/5: 0.49078
Fold 5/5: 0.47692

[XGB 실험 3 결과]
기존 E:             0.47917
E + 미스센스 500개: 0.48190
변화량:             +0.00273
실행 시간:          66.3초
+0.003 기준 충족:   False


# 4. Fold별·클래스별 F1 비교

In [26]:

from sklearn.metrics import f1_score

# 1. Fold별 성능 비교
fold_comparison = pd.DataFrame({
    "Fold": range(1, 6),
    "E_baseline": xgb_fold_scores["E_baseline"],
    "E_plus_type8": xgb_fold_scores["E_plus_type8"],
    "E_plus_multihit2": xgb_fold_scores["E_plus_multihit2"],
    "E_plus_missense500": xgb_fold_scores["E_plus_missense500"]
})

for name in ["E_plus_type8", "E_plus_multihit2", "E_plus_missense500"]:
    fold_comparison[f"{name}_delta"] = (
        fold_comparison[name] - fold_comparison["E_baseline"]
    )

print("[Fold별 XGB Macro F1 비교]")
display(fold_comparison.round(5))

# 2. 클래스별 F1 비교
class_labels = np.arange(len(le.classes_))

class_comparison = pd.DataFrame({
    "class": le.classes_,
    "E_baseline": f1_score(
        y_dev, xgb_oof["E_baseline"],
        labels=class_labels, average=None, zero_division=0
    )
})

for name in ["E_plus_type8", "E_plus_multihit2", "E_plus_missense500"]:
    scores = f1_score(
        y_dev, xgb_oof[name],
        labels=class_labels, average=None, zero_division=0
    )

    class_comparison[name] = scores
    class_comparison[f"{name}_delta"] = (
        scores - class_comparison["E_baseline"]
    )

print("\n[다중 히트 피처 — 클래스별 F1 변화]")
display(
    class_comparison[
        ["class", "E_baseline", "E_plus_multihit2", "E_plus_multihit2_delta"]
    ].sort_values(
        "E_plus_multihit2_delta", ascending=False
    ).round(5)
)

print("\n[피처별 개선/하락 클래스 수]")
for name in ["E_plus_type8", "E_plus_multihit2", "E_plus_missense500"]:
    delta = class_comparison[f"{name}_delta"]
    print(
        f"{name}: "
        f"개선 {(delta > 0).sum()}개 / "
        f"하락 {(delta < 0).sum()}개 / "
        f"동일 {(delta == 0).sum()}개"
    )


[Fold별 XGB Macro F1 비교]


,Fold,E_baseline,E_plus_type8,E_plus_multihit2,E_plus_missense500,E_plus_type8_delta,E_plus_multihit2_delta,E_plus_missense500_delta
0,1,0.49945,0.48661,0.49341,0.49964,-0.01284,-0.00604,0.00020
1,2,0.45525,0.45029,0.46001,0.45566,-0.00496,0.00476,0.00041
2,3,0.47387,0.44708,0.46488,0.46520,-0.02678,-0.00899,-0.00866
3,4,0.47361,0.46668,0.48666,0.49078,-0.00693,0.01306,0.01717
4,5,0.47492,0.47641,0.48941,0.47692,0.00149,0.01449,0.00200



[다중 히트 피처 — 클래스별 F1 변화]


,class,E_baseline,E_plus_multihit2,E_plus_multihit2_delta
5,DLBC,0.31579,0.40000,0.08421
14,LUSC,0.54019,0.57895,0.03875
24,THYM,0.20690,0.23116,0.02426
17,PCPG,0.30675,0.32416,0.01741
8,KIPAN,0.36058,0.37530,0.01472
18,PRAD,0.33040,0.34215,0.01175
25,UCEC,0.63473,0.64596,0.01123
22,TGCT,0.53782,0.54545,0.00764
1,BLCA,0.48521,0.49102,0.00581
2,BRCA,0.55192,0.55741,0.00549



[피처별 개선/하락 클래스 수]
E_plus_type8: 개선 12개 / 하락 14개 / 동일 0개
E_plus_multihit2: 개선 15개 / 하락 11개 / 동일 0개
E_plus_missense500: 개선 16개 / 하락 10개 / 동일 0개


# 5. 다중 히트 피처(실험 2)를 g3 모델에서 검증

In [27]:

# 현재 노트북에 정의된 g3 관련 변수 확인

g3_vars = [
    name for name in globals()
    if "g3" in name.lower()
    or "lgbm" in name.lower()
    or "lightgbm" in name.lower()
]

print("g3 / LightGBM 관련 변수:")
for name in sorted(g3_vars):
    obj = globals()[name]
    print(f"- {name}: {type(obj).__name__}")

if not g3_vars:
    print("현재 노트북에 g3 관련 변수가 없습니다.")


g3 / LightGBM 관련 변수:
현재 노트북에 g3 관련 변수가 없습니다.


In [28]:

import json
from pathlib import Path

notebook_path = (
    ROOT
    / "notebooks"
    / "A_model"
    / "10_lgbm_g3_3model_equal_ensemble.ipynb"
)

with open(notebook_path, "r", encoding="utf-8") as f:
    notebook = json.load(f)

keywords = [
    "LGBMClassifier",
    "lgbm_params",
    "g3",
    "sample_weight",
    "get_folds",
    "run_cv"
]

for idx, cell in enumerate(notebook["cells"]):
    if cell["cell_type"] != "code":
        continue

    source = "".join(cell["source"])

    if any(keyword.lower() in source.lower() for keyword in keywords):
        print(f"\n{'=' * 60}")
        print(f"CELL {idx}")
        print("=" * 60)
        print(source[:5000])



CELL 1
from pathlib import Path
import sys, json, warnings
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score
from xgboost import XGBClassifier

start = Path.cwd().resolve()
ROOT = next((p for p in (start, *start.parents) if (p/'src'/'config.py').is_file() and (p/'scripts'/'fixed_holdout_manifest.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('src/config.py가 있는 프로젝트 폴더에서 실행하세요.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# 노트북의 현재 작업 폴더와 무관하게 프로젝트 스크립트 로딩
import importlib.util
manifest_path = ROOT / 'scripts' / 'fixed_holdout_manifest.py'
spec = importlib.util.spec_from_file_location('fixed_holdout_manifest', manifest_path)
holdout_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(holdout_module)
assert holdout_module.ROOT.

In [29]:

# STEP 25: 기존 g3 피처 변환 코드 확인

import json

notebook_path = (
    ROOT
    / "notebooks"
    / "A_model"
    / "10_lgbm_g3_3model_equal_ensemble.ipynb"
)

with open(notebook_path, "r", encoding="utf-8") as f:
    notebook = json.load(f)

# g3 피처 구성에 필요한 코드 셀
for idx in [5, 6, 8, 9, 10]:
    cell = notebook["cells"][idx]

    if cell["cell_type"] != "code":
        continue

    source = "".join(cell["source"])

    print(f"\n{'=' * 60}")
    print(f"CELL {idx}")
    print("=" * 60)
    print(source)



CELL 5
# 시윤님 03_hotspot_bins.ipynb의 E+Position-bin 함수 재현 (핫스팟 미사용)
import re
from collections import Counter
from scipy.sparse import hstack
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

def make_features_E_sparse(X):
    return csr_matrix(make_E(X).to_numpy(dtype=np.float32))

def extract_position(variant):
    match = re.search(r"\d+", variant)
    return int(match.group()) if match else None
# Position-bin 후보 생성 함수

from collections import defaultdict

def extract_position_bins(X, bin_size=50):
    rows = []
    cols = []
    gene_bins = set()

    for row_idx, (_, row) in enumerate(X.iterrows()):
        patient_bins = set()

        for gene, value in row.items():
            if gene == "ID":
                continue

            if pd.isna(value) or str(value) == "WT":
                continue

            for variant in str(value).split():
                pos = extract_positi

In [30]:

# STEP 26: 팀 기존 g3 코드 재사용

import json
from sklearn.pipeline import Pipeline
from scipy.sparse import hstack
from lightgbm import LGBMClassifier

notebook_path = (
    ROOT
    / "notebooks"
    / "A_model"
    / "10_lgbm_g3_3model_equal_ensemble.ipynb"
)

with open(notebook_path, "r", encoding="utf-8") as f:
    notebook = json.load(f)

# g3 피처 변환에 필요한 코드 셀만 실행
# 전체 노트북 실행은 금지: 홀드아웃/저장 코드 제외
for idx in [5, 7, 8, 9, 10]:
    source = "".join(notebook["cells"][idx]["source"])

    if idx == 5:
        # 해당 셀의 LR 파이프라인 생성 부분은 제외
        source = source.split("pipeline_E_position_LR =")[0]

    exec(
        compile(source, f"team_g3_cell_{idx}", "exec"),
        globals()
    )

print("g3 모델 설정 완료")
print("모델:", type(lgbm_g3.named_steps["model"]).__name__)
print("피처 변환기:", type(lgbm_g3.named_steps["features"]).__name__)
print("Estimators:", G3_PARAMS["n_estimators"])
print("Learning rate:", G3_PARAMS["learning_rate"])
print("Num leaves:", G3_PARAMS["num_leaves"])
print("Class weight:", G3_PARAMS["class_weight"])


g3 피처 변환기 준비 완료
g3 모델 설정 완료
모델: LGBMClassifier
피처 변환기: FeaturesEHotspot3BinExact
Estimators: 800
Learning rate: 0.02
Num leaves: 15
Class weight: balanced


In [31]:

# STEP 27: 기존 g3 Baseline 검증

import time

start = time.time()

g3_baseline_oof = np.full(len(y_dev), -1, dtype=int)
g3_baseline_proba = np.zeros(
    (len(y_dev), len(le.classes_)),
    dtype=np.float32
)
g3_baseline_fold_scores = []
seen = np.zeros(len(y_dev), dtype=int)

for fold_num, (train_idx, val_idx) in enumerate(team_folds, start=1):
    model = clone(lgbm_g3)

    # 피처 선택은 각 Fold의 학습 데이터에서만 수행
    model.fit(
        X_dev_raw.iloc[train_idx],
        y_dev[train_idx]
    )

    proba = model.predict_proba(
        X_dev_raw.iloc[val_idx]
    )

    pred = proba.argmax(axis=1)

    g3_baseline_proba[val_idx] = proba
    g3_baseline_oof[val_idx] = pred
    seen[val_idx] += 1

    score = f1_score(
        y_dev[val_idx],
        pred,
        average="macro",
        zero_division=0
    )

    g3_baseline_fold_scores.append(score)

    print(f"Fold {fold_num}/5: {score:.5f}")

assert np.all(seen == 1), "OOF 검증 행 누락 또는 중복"
assert np.isfinite(g3_baseline_proba).all()

g3_baseline_f1 = f1_score(
    y_dev,
    g3_baseline_oof,
    average="macro",
    zero_division=0
)

print("\n[g3 Baseline 결과]")
print(f"OOF Macro F1: {g3_baseline_f1:.5f}")
print(f"Fold 평균:    {np.mean(g3_baseline_fold_scores):.5f}")
print(f"Fold 표준편차: {np.std(g3_baseline_fold_scores):.5f}")
print(f"실행 시간:   {time.time() - start:.1f}초")


Fold 1/5: 0.51368
Fold 2/5: 0.49019
Fold 3/5: 0.49389
Fold 4/5: 0.50619
Fold 5/5: 0.50141

[g3 Baseline 결과]
OOF Macro F1: 0.50465
Fold 평균:    0.50107
Fold 표준편차: 0.00843
실행 시간:   1113.8초


In [32]:

# STEP 28: g3 + 다중 히트 2개

import time
from scipy.sparse import csr_matrix, hstack

class FeaturesG3WithMultihit(FeaturesEHotspot3BinExact):
    def transform(self, X):
        # 기존 g3 피처
        X_base = super().transform(X)

        # 개발셋에서 해당 환자의 다중 히트 피처 가져오기
        X_extra = X_multihit_log.iloc[X.index].to_numpy(
            dtype=np.float32
        )

        return hstack(
            [X_base, csr_matrix(X_extra)],
            format="csr",
            dtype=np.float32
        )

g3_multihit_model = Pipeline([
    (
        "features",
        FeaturesG3WithMultihit(
            bin_size=50,
            min_patients=5,
            hotspot_multiplier=3
        )
    ),
    ("model", LGBMClassifier(**G3_PARAMS))
])

start = time.time()

g3_multihit_oof = np.full(len(y_dev), -1, dtype=int)
g3_multihit_proba = np.zeros(
    (len(y_dev), len(le.classes_)),
    dtype=np.float32
)
g3_multihit_fold_scores = []
seen = np.zeros(len(y_dev), dtype=int)

for fold_num, (train_idx, val_idx) in enumerate(team_folds, start=1):
    model = clone(g3_multihit_model)

    # 원본 개발셋의 행 인덱스를 유지
    X_train = X_dev_raw.iloc[train_idx].copy()
    X_val = X_dev_raw.iloc[val_idx].copy()

    X_train.index = train_idx
    X_val.index = val_idx

    model.fit(X_train, y_dev[train_idx])

    proba = model.predict_proba(X_val)
    pred = proba.argmax(axis=1)

    g3_multihit_proba[val_idx] = proba
    g3_multihit_oof[val_idx] = pred
    seen[val_idx] += 1

    score = f1_score(
        y_dev[val_idx],
        pred,
        average="macro",
        zero_division=0
    )

    g3_multihit_fold_scores.append(score)
    print(f"Fold {fold_num}/5: {score:.5f}")

assert np.all(seen == 1)
assert np.isfinite(g3_multihit_proba).all()

g3_multihit_f1 = f1_score(
    y_dev,
    g3_multihit_oof,
    average="macro",
    zero_division=0
)

delta = g3_multihit_f1 - g3_baseline_f1

print("\n[g3 다중 히트 실험 결과]")
print(f"기존 g3:            {g3_baseline_f1:.5f}")
print(f"g3 + 다중 히트 2개: {g3_multihit_f1:.5f}")
print(f"변화량:             {delta:+.5f}")
print(f"실행 시간:          {time.time() - start:.1f}초")
print(f"+0.003 기준 충족:   {delta >= 0.003}")


Fold 1/5: 0.51407
Fold 2/5: 0.47642
Fold 3/5: 0.49061
Fold 4/5: 0.49477
Fold 5/5: 0.50417

[g3 다중 히트 실험 결과]
기존 g3:            0.50465
g3 + 다중 히트 2개: 0.49977
변화량:             -0.00488
실행 시간:          1120.0초
+0.003 기준 충족:   False


# 6. 실험 결과 정리

### 1. 실험 목적

기존 E 피처에 새로운 변이 관련 피처를 추가했을 때
암종 분류 모델의 OOF Macro F1이 개선되는지 검증한다.

- 데이터: 개발셋 4,960명
- 검증: 팀 공통 5-Fold
- 평가 지표: OOF Macro F1
- 홀드아웃: 미사용

### 2. 실험 피처

| 실험 | 추가 피처 | 개수 |
|---|---|---:|
| 1 | 환자별 변이 유형 비율, 엔트로피, 다양성 등 | 8 |
| 2 | 다중 변이 유전자 수, 유전자별 최대 변이 수 | 2 |
| 3 | 유전자별 미스센스 변이 여부 (상위 500개 유전자) | 500 |

### 3. 모델별 OOF Macro F1

| 피처 구성 | LR | XGB | g3 |
|---|---:|---:|---:|
| 기존 E | 0.48485 | 0.47917 | 0.50465 |
| E + 변이 유형 8개 | 0.48323 | 0.46900 | 미검증 |
| E + 다중 히트 2개 | 0.48774 | 0.48317 | 0.49977 |
| E + 미스센스 500개 | 0.48079 | 0.48190 | 미검증 |

### 4. 기존 E 대비 성능 변화

| 추가 피처 | LR Δ | XGB Δ | g3 Δ |
|---|---:|---:|---:|
| 변이 유형 8개 | -0.00162 | -0.01017 | - |
| 다중 히트 2개 | +0.00289 | +0.00400 | -0.00488 |
| 미스센스 500개 | -0.00406 | +0.00273 | - |

### 5. 주요 결과

**변이 유형 8개**
- LR과 XGB 모두 성능 하락
- 현재 설정에서는 추가 효과 확인되지 않음

**다중 히트 2개**
- LR: +0.00289
- XGB: +0.00400
- g3: -0.00488
- XGB에서 단일 모델 개선 기준(+0.003) 충족
- XGB 5개 Fold 중 3개 개선, 2개 하락
- XGB 26개 암종 중 15개 개선, 11개 하락
- 모델에 따라 효과가 달라짐

**미스센스 500개**
- LR에서는 성능 하락
- XGB에서는 +0.00273 개선
- 단일 모델 개선 기준(+0.003)에는 미달
- 상위 500개 유전자 선정은 각 학습 Fold에서만 수행

### 6. 결론

- 가장 유망한 추가 피처는 다중 히트 2개
- XGB에만 적용하는 방안을 후속 검증 후보로 제안
- g3에서는 성능이 하락했으므로 적용 보류
- Fold별 성능 변동이 있어 일반화 효과는 추가 검증 필요
- 앙상블 개선 여부는 아직 검증하지 않음
- 최종 모델 채택 여부는 별도 앙상블 검증 후 판단

### 7. 실험 범위 및 한계

- 모든 실험은 개발셋 OOF 기준
- 홀드아웃 및 리더보드 성능 검증 없음
- 단일 모델 개선이 앙상블 개선을 보장하지 않음
- XGB 추가 피처 실험의 OOF 확률값은 저장하지 않음
